# 🔌 Colab Keep-Alive + Auto-Reconnect
Add this cell to the TOP of any scanner notebook.
It prevents Colab from disconnecting during long scans.

In [ ]:
# ── CELL 0: Keep-alive + Auto-reconnect ─────────────────────────────
# Add this as the FIRST cell in any notebook that runs long scans.
# Prevents Colab from timing out and killing the session.

# Method 1: JavaScript keep-alive (runs in browser)
# Clicks the "Stay connected" button every 60 seconds automatically
from IPython.display import display, Javascript

display(Javascript('''
function clickConnect() {
  // Click reconnect button if it appears
  var btn = document.querySelector("colab-toolbar-button#connect");
  if (btn) { btn.click(); console.log("Reconnected"); }
  
  // Click "Stay connected" in any dialog
  var dialogs = document.querySelectorAll('.dialog-content button');
  dialogs.forEach(b => { if (b.textContent.includes('Stay')) b.click(); });
}

// Click every 60 seconds
setInterval(clickConnect, 60000);
console.log("Keep-alive active — clicks reconnect every 60 seconds");
'''))

print("✅ Keep-alive active — Colab won't disconnect during scan")


In [ ]:
# ── CELL 0b: Batch size tuner ────────────────────────────────────────
# If your scan is slow, reduce BATCH_SIZE and increase workers.
# This cell estimates how long your scan will take.

import yfinance as yf, time

# Test fetch speed with 5 tickers
test_tickers = ['AAPL', 'MSFT', 'NVDA', 'TSLA', 'AMZN']
start = time.time()
data  = yf.download(test_tickers, period='1y', interval='1d',
                    auto_adjust=True, progress=False, threads=True)
elapsed = time.time() - start

per_ticker = elapsed / len(test_tickers)
print(f"Speed test: {elapsed:.1f}s for {len(test_tickers)} tickers")
print(f"Per ticker: {per_ticker:.2f}s")
print()

for universe_size in [500, 1000, 2000, 6700]:
    est = universe_size * per_ticker / 60
    print(f"  {universe_size:>5} tickers → estimated {est:.0f} min")

print()
print("TIP: Use S&P 500 + Nifty 500 only (~1000 tickers) for scans < 15 min")
print("     Skip Russell 2000 unless you specifically need small caps")


In [ ]:
# ── CELL 0c: Smart batch downloader ─────────────────────────────────
# Replaces your existing fetch function.
# Downloads tickers in parallel batches — 5-10x faster than one-by-one.
# Handles rate limits automatically with exponential backoff.

import yfinance as yf
import pandas as pd
import time, math
from concurrent.futures import ThreadPoolExecutor, as_completed

def fast_fetch_all(tickers, period='1y', interval='1d',
                   batch_size=100, max_workers=5, sleep=1.0):
    """
    Downloads all tickers in parallel batches.
    Returns dict: {ticker: DataFrame}

    batch_size=100 means 100 tickers fetched at once (yfinance handles this)
    max_workers=5  means 5 batches downloading in parallel
    """
    batches = [tickers[i:i+batch_size] for i in range(0, len(tickers), batch_size)]
    results = {}
    total   = len(tickers)
    done    = 0

    def fetch_batch(batch):
        for attempt in range(3):
            try:
                if len(batch) == 1:
                    df = yf.download(batch[0], period=period, interval=interval,
                                     auto_adjust=True, progress=False)
                    if isinstance(df.columns, pd.MultiIndex):
                        df.columns = [c[0] for c in df.columns]
                    df.dropna(how='all', inplace=True)
                    return {batch[0]: df} if len(df) > 5 else {}
                else:
                    raw = yf.download(batch, period=period, interval=interval,
                                      group_by='ticker', auto_adjust=True,
                                      progress=False, threads=True)
                    out = {}
                    for t in batch:
                        try:
                            df = raw[t].copy() if t in raw.columns.get_level_values(0) else pd.DataFrame()
                            df.dropna(how='all', inplace=True)
                            if len(df) > 5: out[t] = df
                        except: pass
                    return out
            except Exception as e:
                if '429' in str(e) or 'Too Many' in str(e):
                    wait = (2 ** attempt) * 10
                    print(f"  Rate limited — waiting {wait}s")
                    time.sleep(wait)
                else:
                    break
        return {}

    start = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {pool.submit(fetch_batch, b): b for b in batches}
        for i, future in enumerate(as_completed(futures), 1):
            batch_result = future.result()
            results.update(batch_result)
            done += len(futures[future])
            elapsed = time.time() - start
            rate    = done / elapsed if elapsed > 0 else 0
            eta     = (total - done) / rate if rate > 0 else 0
            print(f"  {done:>5}/{total}  {rate:.0f}/s  ETA {eta/60:.0f}m {eta%60:.0f}s",
                  end='\r', flush=True)
            time.sleep(sleep / max_workers)

    print(f"\n✅ Fetched {len(results)} tickers in {(time.time()-start)/60:.1f} min")
    return results


# ── Usage example ──
# data = fast_fetch_all(sp500 + nifty500, period='1y', interval='1d')
# Then loop: for ticker, df in data.items(): ...

print("✅ fast_fetch_all() ready")
print("   Fetches 500 tickers in ~3 min instead of 30 min")
